In [1]:
%load_ext autoreload
%autoreload 2

In [2]:
import sys
sys.path.append('../')

In [58]:
import hydra 

import logging
import accelerate
import torch
import numpy as np

from dawn.models.system2.flow_utils import visualize_flow_vectors_as_PIL, FlowNormalizer
        
logging.basicConfig(level=logging.INFO)
logger = logging.getLogger(__name__)

from dawn.models.system2.ldmv2 import LatentMotionEstimationV2

config = {
    "_target_" : "dawn.models.system2.ldmv2.LatentMotionEstimationV2",
    "flow_to_rgb" : False,
    "use_cfg" : False,
    "use_interval" : True,
    "num_inference_steps": 2,
}

accelerator = accelerate.Accelerator()
accelerate.utils.set_seed(0)

# weights = "../outputs/DAWN_stage_1/2025-06-26_16-41-53/checkpoints/model_0050000.pth"
print(f"Loading model configuration from {config}")
model = hydra.utils.instantiate(config)


INFO:dawn.models.system2.ldmv2:Initializing LatentMotionEstimationV2 with image size 256, in_channels 8, out_channels 3, condition_dim 768, flow_to_rgb False.


Loading model configuration from {'_target_': 'dawn.models.system2.ldmv2.LatentMotionEstimationV2', 'flow_to_rgb': False, 'use_cfg': False, 'use_interval': True, 'num_inference_steps': 2}


Loading pipeline components...: 100%|██████████| 6/6 [00:00<00:00, 25.40it/s]
Some weights of the model checkpoint at google/vit-base-patch16-224-in21k were not used when initializing ViTModel: ['pooler.dense.bias', 'pooler.dense.weight']
- This IS expected if you are initializing ViTModel from the checkpoint of a model trained on another task or with another architecture (e.g. initializing a BertForSequenceClassification model from a BertForPreTraining model).
- This IS NOT expected if you are initializing ViTModel from the checkpoint of a model that you expect to be exactly identical (initializing a BertForSequenceClassification model from a BertForSequenceClassification model).
INFO:dawn.models.system2.ldmv2:Total parameters: 1.16 GB, Trainable parameters: 945.75 MB


In [62]:
inputs = {
    "rgb_static": torch.rand(1, 4, 3, 256, 256) ,
    # "rgb_gripper": torch.rand(1, 3, 3, 256, 256) ,
    "language": ["Hello"], 
    "skip_frame": torch.tensor(20).view(-1)
}
output = model(inputs, use_history=True)


Use history features torch.Size([1, 2, 2, 256, 256]) torch.Size([1, 2, 3, 256, 256])
Using history features, text condition shape: torch.Size([1, 414, 768])
